
# Shareable interactive notebook

This version is designed to be easier to share with others.

It uses **ipywidgets** instead of Matplotlib's built-in `Slider`, which makes it much more reliable when publishing through **Voilà** or **Binder**.

## Best way to publish

1. Put this notebook in a GitHub repository.
2. Add the `requirements.txt` file from this folder.
3. Share either:
   - the **GitHub link** for viewing the notebook, or
   - a **Binder** link for running it interactively, or
   - a **Voilà** deployment if you want it to look like a small app.

## Notes

- The PDE is solved once using the parameters in the simulation cell.
- The sliders control only the **overlay function** drawn on top of the precomputed data.
- If you want users to also change `p, q, p_s, q_s, L`, that is possible too, but then the notebook would need to **recompute the PDE**, which is much heavier.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks
from scipy.integrate import solve_ivp
from numba import njit
import ipywidgets as widgets
from IPython.display import display


In [ ]:

def getRho_crit(p, q, p_s, q_s):
    rc1 = (q_s * (p - q)) / ((p * q_s) - (q * p_s))
    rc2 = (p_s * (p - q)) / ((p * q_s) - (q * p_s))
    return rc1, rc2

@njit
def dphi_dt(t, phi, p, q, p_s, q_s):
    L = len(phi)
    dphi = np.zeros(L)

    vp = p * (1 - phi[0]) + p_s * phi[0]
    vm = q * (1 - phi[L - 1]) + q_s * phi[L - 1]

    dphi[1:L-1] = (
        phi[:-2] - 2 * phi[1:-1] + phi[2:]
        + vp * (phi[2:L] - phi[1:L-1])
        - vm * (phi[1:L-1] - phi[0:L-2])
    )

    dphi[0] = (
        (1 - phi[0]) * (q_s * phi[L - 1] + (1 + p) * phi[1])
        - (1 + p_s) * phi[0] * (1 - phi[1])
        - q * phi[0] * (1 - phi[L - 1])
    )

    dphi[L - 1] = (
        (1 - phi[L - 1]) * (p_s * phi[0] + (1 + q) * phi[L - 2])
        - (1 + q_s) * phi[L - 1] * (1 - phi[L - 2])
        - p * phi[L - 1] * (1 - phi[0])
    )

    return dphi


In [ ]:

# Simulation parameters
p = 1.0
q = 0.1
p_s = 0.1
q_s = 1.0
L = 6000

rho = getRho_crit(p, q, p_s, q_s)[1]
phi = np.ones(L) * rho

step_size = 1e4
t_eval = np.arange(4e4, 1e5 + step_size, step_size)
t_span = (0, float(np.max(t_eval)))

# Solve once
sol = solve_ivp(
    dphi_dt,
    t_span,
    phi,
    args=(p, q, p_s, q_s),
    method='LSODA',
    t_eval=t_eval,
)

density = sol.y
print(f"Computed density with shape {density.shape}")


In [ ]:

def f(z, t, v0, A, s):
    z = np.asarray(z, dtype=float)
    arg = v0 + (2.0 * z / 3.0)

    out = np.full_like(z, np.nan, dtype=float)
    valid = arg > 0

    out[valid] = (
        A
        * (1.0 / np.sqrt(arg[valid]))
        * np.exp(-(1.0 / (s * t**(1/3))) * (1.0 / (arg[valid]**2)))
        * np.exp(-(arg[valid]**2) / s)
    )

    out[~np.isfinite(out)] = np.nan
    return out


def make_plot(v0=1.237, A=0.1765, s=5.98, plot_every=2, scatter_every=35, offset=1250):
    fig, ax = plt.subplots(figsize=(14, 6), dpi=140)

    plot_times = range(0, len(t_eval), int(plot_every))
    norm = LogNorm(vmin=t_eval[plot_times[0]], vmax=t_eval[plot_times[-1]] * 1.15)
    cm = plt.get_cmap("viridis")

    dxl = L - 400
    xl = np.arange(-dxl, 0)

    for ti in plot_times:
        t = t_eval[ti]
        y = density[:, ti]

        maxima_idx, _ = find_peaks(rho - y[-dxl:])
        if len(maxima_idx) == 0:
            continue

        start_idx = max(maxima_idx[0] - int(offset), 0)

        x_fit = xl[start_idx:] / np.power(t, 2/3)
        y_fit = (rho - y[-dxl:])[start_idx:] * np.power(t, 1/3)

        x_fit = x_fit[::int(scatter_every)]
        y_fit = y_fit[::int(scatter_every)]

        ax.plot(
            x_fit,
            y_fit,
            alpha=1,
            linewidth=1,
            color=cm(norm(t)),
            linestyle="none",
            zorder=1,
            marker="o",
            markersize=1,
        )

    ax.set_xlabel("$z$", fontsize=14)
    ax.set_ylabel(r"$[\overline{ho} - ho(z)] \cdot t^{1/3}$", fontsize=14)
    ax.invert_xaxis()
    ax.set_title(
        f"L = {L:,}, mean rho = {rho:.4f}, p={p}, q={q}, p'={p_s}, q'={q_s}",
        fontsize=12,
    )

    z = np.linspace(min(ax.get_xlim()), max(ax.get_xlim()), 1000)
    for ti in plot_times:
        t = t_eval[ti]
        ax.plot(z, f(z, t, v0, A, s), linewidth=1, color=cm(norm(t)), zorder=2)

    plt.show()


In [ ]:

controls = {
    'v0': widgets.FloatSlider(value=1.237, min=0.05, max=3.0, step=0.01, description='v0'),
    'A': widgets.FloatSlider(value=0.1765, min=0.01, max=2.0, step=0.005, description='A'),
    's': widgets.FloatSlider(value=5.98, min=0.05, max=10.0, step=0.01, description='s'),
    'plot_every': widgets.IntSlider(value=2, min=1, max=max(1, len(t_eval)-1), step=1, description='plot every'),
    'scatter_every': widgets.IntSlider(value=35, min=1, max=100, step=1, description='scatter every'),
    'offset': widgets.IntSlider(value=1250, min=0, max=max(1500, L-400), step=25, description='offset'),
}

ui = widgets.VBox([
    widgets.HTML('<h3>Adjust overlay parameters</h3>'),
    controls['v0'],
    controls['A'],
    controls['s'],
    controls['plot_every'],
    controls['scatter_every'],
    controls['offset'],
])

out = widgets.interactive_output(make_plot, controls)
display(ui, out)
